# OneVoice V2 — Đánh giá SenseVoice English đã fine-tune

Notebook này không train và không thay runtime ONNX. Nó benchmark checkpoint cuối model.pt.ep10 trên test clean/noisy. Cần GPU. Không dùng model.pt.avg3.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, os, subprocess, sys

GITHUB_REPO = 'https://github.com/Platypus27-coder/OneVoice.git'
BRANCH = 'main'
REPO = Path('/content/OneVoice')
MYDRIVE = Path('/content/drive/MyDrive')
WORK_ROOT = MYDRIVE / 'OneVoice'
MANIFEST = MYDRIVE / 'onevoice_audio_v2_1/manifest.jsonl'
CHECKPOINT = WORK_ROOT / 'models/sensevoice_en_construction_v1/model.pt.ep10'
REPORT_ROOT = WORK_ROOT / 'reports/en_asr_finetuned_v1'

if (REPO / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, GITHUB_REPO, str(REPO)], check=True)
os.chdir(REPO)
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['MODELSCOPE_CACHE'] = str(WORK_ROOT / 'model_cache/modelscope')
# Install a matched CUDA 13.0 pair. Installing bare torch can leave Colab's older torchaudio linked against another CUDA version.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--force-reinstall', '--no-cache-dir', 'torch==2.9.1', 'torchaudio==2.9.1', '--index-url', 'https://download.pytorch.org/whl/cu130'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'numpy==2.2.6', 'funasr>=1.4.3', 'modelscope', 'soundfile'], check=True)
import torch, torchaudio
if not torch.cuda.is_available():
    raise RuntimeError('Chọn GPU runtime trước khi đánh giá checkpoint SenseVoice.')
if not MANIFEST.is_file() or not CHECKPOINT.is_file():
    raise FileNotFoundError(f'Missing manifest or final checkpoint: {MANIFEST}, {CHECKPOINT}')
print('GPU:', torch.cuda.get_device_name(0))
print('Checkpoint:', CHECKPOINT, f'({CHECKPOINT.stat().st_size / 1024**2:.1f} MB)')


In [ ]:
# Smoke test: kiểm tra checkpoint có nạp và decode được trước khi chạy toàn bộ test.
smoke_dir = REPORT_ROOT / 'smoke_clean'
if not (smoke_dir / 'aggregate.json').is_file():
    subprocess.run([sys.executable, 'scripts/benchmark_sensevoice_checkpoint.py', str(MANIFEST), '--checkpoint', str(CHECKPOINT), '--audio', 'clean', '--split', 'test', '--max-samples', '16', '--progress-every', '4', '--report-dir', str(smoke_dir)], check=True)
else:
    print('Smoke test already completed; skipping.')


In [ ]:
# Full held-out test. Existing complete reports are retained; rerun safely after a runtime/account change.
for audio in ('clean', 'noisy'):
    report_dir = REPORT_ROOT / audio
    if all((report_dir / name).is_file() for name in ('aggregate.json', 'predictions.csv', 'run_manifest.json')):
        print(f'{audio} report already complete; skipping.')
        continue
    subprocess.run([sys.executable, 'scripts/benchmark_sensevoice_checkpoint.py', str(MANIFEST), '--checkpoint', str(CHECKPOINT), '--audio', audio, '--split', 'test', '--progress-every', '25', '--report-dir', str(report_dir)], check=True)


In [ ]:
results = {audio: json.loads((REPORT_ROOT / audio / 'aggregate.json').read_text(encoding='utf-8')) for audio in ('clean', 'noisy')}
display(results)
print('Chỉ export/replace ONNX nếu WER/CER, critical-term recall và clean/noisy regression tốt hơn baseline.')
